# Classical Sentiment Baselines

Baseline models for 3-class Twitter sentiment classification. These establish
the performance floor that transformer models (RoBERTa) should beat.

**Models:**
1. Bag of Words + Logistic Regression
2. Bag of Words + Multinomial Naive Bayes
3. TF-IDF + Linear SVM
4. TF-IDF + Random Forest
5. Word2Vec + Logistic Regression

**Metrics:** Accuracy, F1 (macro), Precision (macro), Recall (macro), QWK

All models use the sentiment-aware preprocessing from `preprocessing/preprocessing.py`.

In [ ]:
import sys, os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    cohen_kappa_score,
)

warnings.filterwarnings("ignore", category=FutureWarning)

# Allow imports from project root
sys.path.insert(0, os.path.abspath(".."))

SEED = 42
np.random.seed(SEED)

## 1. Load & Preprocess Data

In [ ]:
from preprocessing.preprocessing import preprocess_df

# Load the training set (same data used for RoBERTa)
df = pd.read_csv("../dataset/train.csv", encoding="cp1252")
df = df[["textID", "text", "sentiment"]].dropna(subset=["text", "sentiment"])
df = df[df["text"].str.strip() != ""].reset_index(drop=True)

# Encode labels
label_map = {"negative": 0, "neutral": 1, "positive": 2}
label_names = ["negative", "neutral", "positive"]
df["label"] = df["sentiment"].map(label_map)

# Apply dual preprocessing
df = preprocess_df(df, text_col="text")

print(f"Dataset size: {len(df)}")
print(f"Class distribution:\n{df['sentiment'].value_counts()}")
print(f"\nSample preprocessing:")
print(df[["text", "clean_text", "light_text"]].head(3).to_string())

In [ ]:
# Stratified train/test split — 80/20, same as sentiment_analysis experiments
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"],
    df["label"],
    test_size=0.2,
    random_state=SEED,
    stratify=df["label"],
)

print(f"Train: {len(X_train)}, Test: {len(X_test)}")
print(f"Train label distribution: {dict(y_train.value_counts().sort_index())}")
print(f"Test label distribution:  {dict(y_test.value_counts().sort_index())}")

## 2. Feature Extraction

In [ ]:
# Bag of Words
bow_vec = CountVectorizer(max_features=50000, ngram_range=(1, 2))
X_train_bow = bow_vec.fit_transform(X_train)
X_test_bow = bow_vec.transform(X_test)
print(f"BoW features: {X_train_bow.shape[1]}")

# TF-IDF
tfidf_vec = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), sublinear_tf=True)
X_train_tfidf = tfidf_vec.fit_transform(X_train)
X_test_tfidf = tfidf_vec.transform(X_test)
print(f"TF-IDF features: {X_train_tfidf.shape[1]}")

## 3. Evaluation Utilities

In [ ]:
results = []

def evaluate_model(name, y_true, y_pred):
    """Compute metrics and store results. Returns a dict of metrics."""
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    qwk = cohen_kappa_score(y_true, y_pred, weights="quadratic")

    row = {
        "model": name,
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1_macro": f1,
        "qwk": qwk,
    }
    results.append(row)

    print(f"\n{'=' * 50}")
    print(f"{name}")
    print(f"{'=' * 50}")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f} (macro)")
    print(f"Recall:    {rec:.4f} (macro)")
    print(f"F1:        {f1:.4f} (macro)")
    print(f"QWK:       {qwk:.4f}")
    print(f"\n{classification_report(y_true, y_pred, target_names=label_names, digits=4)}")
    return row

## 4. Model Training & Evaluation

### 4a. Bag of Words + Logistic Regression

In [ ]:
lr_bow = LogisticRegression(max_iter=1000, random_state=SEED, C=1.0)
lr_bow.fit(X_train_bow, y_train)
y_pred_lr_bow = lr_bow.predict(X_test_bow)
evaluate_model("BoW + Logistic Regression", y_test, y_pred_lr_bow);

### 4b. Bag of Words + Multinomial Naive Bayes

In [ ]:
nb_bow = MultinomialNB(alpha=1.0)
nb_bow.fit(X_train_bow, y_train)
y_pred_nb_bow = nb_bow.predict(X_test_bow)
evaluate_model("BoW + Naive Bayes", y_test, y_pred_nb_bow);

### 4c. TF-IDF + Linear SVM

In [ ]:
svm_tfidf = LinearSVC(max_iter=2000, random_state=SEED, C=1.0)
svm_tfidf.fit(X_train_tfidf, y_train)
y_pred_svm_tfidf = svm_tfidf.predict(X_test_tfidf)
evaluate_model("TF-IDF + Linear SVM", y_test, y_pred_svm_tfidf);

### 4d. TF-IDF + Random Forest

In [ ]:
rf_tfidf = RandomForestClassifier(
    n_estimators=200, random_state=SEED, n_jobs=-1
)
rf_tfidf.fit(X_train_tfidf, y_train)
y_pred_rf_tfidf = rf_tfidf.predict(X_test_tfidf)
evaluate_model("TF-IDF + Random Forest", y_test, y_pred_rf_tfidf);

### 4e. Word2Vec + Logistic Regression

Train Word2Vec on our corpus, then mean-pool token embeddings per tweet.

In [ ]:
from gensim.models import Word2Vec

# Tokenize for Word2Vec
train_tokens = [text.split() for text in X_train]
test_tokens = [text.split() for text in X_test]

# Train Word2Vec
w2v_model = Word2Vec(
    sentences=train_tokens,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    epochs=5,
    seed=SEED,
)
print(f"Word2Vec vocabulary size: {len(w2v_model.wv)}")
print(f"Embedding dimension: {w2v_model.wv.vector_size}")

In [ ]:
def text_to_w2v(tokens, model, dim=100):
    """Mean-pool Word2Vec embeddings for a list of tokens."""
    vocab = model.wv.key_to_index
    vecs = [model.wv[t] for t in tokens if t in vocab]
    if vecs:
        return np.mean(vecs, axis=0)
    return np.zeros(dim)

X_train_w2v = np.array([text_to_w2v(t, w2v_model) for t in train_tokens])
X_test_w2v = np.array([text_to_w2v(t, w2v_model) for t in test_tokens])

print(f"W2V train shape: {X_train_w2v.shape}")
print(f"W2V test shape: {X_test_w2v.shape}")

# Train LR on W2V features
lr_w2v = LogisticRegression(max_iter=1000, random_state=SEED, C=1.0)
lr_w2v.fit(X_train_w2v, y_train)
y_pred_lr_w2v = lr_w2v.predict(X_test_w2v)
evaluate_model("Word2Vec + Logistic Regression", y_test, y_pred_lr_w2v);

## 5. Results Summary

In [ ]:
results_df = pd.DataFrame(results).sort_values("f1_macro", ascending=False).reset_index(drop=True)
print(results_df.round(4).to_string(index=False))

In [ ]:
# Bar chart of F1 macro scores
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(results_df["model"], results_df["f1_macro"], color="steelblue")
ax.set_xlabel("F1 (macro)")
ax.set_title("Classical Baseline: F1 Macro Scores")
ax.invert_yaxis()

for bar, val in zip(bars, results_df["f1_macro"]):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height() / 2,
            f"{val:.4f}", va="center", fontsize=10)

plt.tight_layout()
plt.show()

## 6. Confusion Matrices

In [ ]:
model_preds = {
    "BoW + LR": y_pred_lr_bow,
    "BoW + NB": y_pred_nb_bow,
    "TF-IDF + SVM": y_pred_svm_tfidf,
    "TF-IDF + RF": y_pred_rf_tfidf,
    "W2V + LR": y_pred_lr_w2v,
}

fig, axes = plt.subplots(1, len(model_preds), figsize=(5 * len(model_preds), 4))

for ax, (name, y_pred) in zip(axes, model_preds.items()):
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1, 2])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=label_names, yticklabels=label_names)
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

plt.tight_layout()
plt.show()

## 7. QWK vs F1 Comparison

QWK (Quadratic Weighted Kappa) penalizes distant misclassifications more heavily.
For ordinal sentiment (negative < neutral < positive), predicting negative as positive
is worse than predicting negative as neutral. F1 macro treats both errors equally.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(results_df["f1_macro"], results_df["qwk"], s=100, c="steelblue", zorder=5)
for _, row in results_df.iterrows():
    ax.annotate(row["model"], (row["f1_macro"], row["qwk"]),
                textcoords="offset points", xytext=(8, 4), fontsize=9)

ax.set_xlabel("F1 (macro)")
ax.set_ylabel("QWK")
ax.set_title("F1 Macro vs QWK — Classical Baselines")
ax.plot([0, 1], [0, 1], "--", color="gray", alpha=0.3)  # reference diagonal
ax.set_xlim(ax.get_xlim()[0] - 0.01, ax.get_xlim()[1] + 0.03)
plt.tight_layout()
plt.show()

## 8. Discussion

**Key findings:**

- **BoW + Logistic Regression** is expected to be the strongest single classical model,
  benefiting from the sentiment-aware stopword list.
- **TF-IDF + SVM** is competitive but LR tends to edge it out — likely because LR
  handles the larger feature space (with preserved negation words) better.
- **Random Forest** underperforms on sparse text features (known limitation).
- **Word2Vec** captures semantic similarity but loses the discriminative power of
  individual sentiment-bearing tokens after mean-pooling.

**QWK vs F1:** Models that confuse negative↔neutral (adjacent) should rank higher on
QWK than models that confuse negative↔positive (distant). Check whether the confusion
matrices above confirm this.

**Context for transformer comparison:** The RoBERTa fine-tuned model in
`sentiment/roberta_base.py` should significantly outperform these baselines. The gap
quantifies what pretrained language understanding adds beyond bag-of-words features.